# Base modelling

In [ ]:
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, confusion_matrix,
    accuracy_score, f1_score, recall_score, precision_score,
    average_precision_score, brier_score_loss
)
# ../ navigates up from notebooks/ to the repo root
df = pd.read_parquet("../data/individual_merges/parquet_saves/macro_ind_robust_me.parquet").drop(columns=['credit_gap_missing'], errors='ignore')  # final run: flag excluded, not reclassified
print(df.columns.tolist())

In [ ]:
country_col = 'country_iso'
target = 'crisis'
id_cols = [country_col, 'year', target]
feat_cols = [col for col in df.columns if col not in id_cols]

# Collinearity identification

## VIF

In [ ]:
def compute_vif(df, cols):
    tmp = df[cols].dropna() #selects only columns you want to check, and drops na since it cannot handle missing values
    vif_data = pd.DataFrame({
        'feature': cols,
        'VIF': [variance_inflation_factor(tmp.values, i) for i in range(len(cols))] #takes full feature matrix and index of column to check, then returns VIF score
    })
    return vif_data.sort_values('VIF', ascending=False)

In [ ]:
print("\n=== Macro ===")
print(compute_vif(df, feat_cols))

BIS_REER > 10, thus is dropped

In [ ]:
feat_cols.remove("BIS_REER")
feat_cols.remove('FM.AST.NFRG.CN') #dropped as it is only used to compute a derived variable

# Expanding window split

In [ ]:
# A9: shared with 11a-d and 12a/b in ews_common.py, so a fix is made once for every specification.
from ews_common import expanding_window_split, calibration_split

In [ ]:
splits = expanding_window_split(df, min_train_years=5)
# A1: the alarm cut-off for test year t is chosen on the predictions for t-1. The first test year has
# no evaluated predecessor, so one extra window (one training year shorter) is fitted to calibrate it.
calibration = calibration_split(df, min_train_years=5)
print(f"Calibration window: train {calibration[0]['year'].min()}-{calibration[0]['year'].max()}, "
      f"predict {calibration[1]['year'].values[0]} (for the first cut-off only)")
for train, test in splits[:3]:
    print(f"Train: {train['year'].min()}-{train['year'].max()}, Test: {test['year'].values[0]}")

# Results dataframe

In [ ]:
# A9: shared with 11a-d and 12a/b in ews_common.py, so a fix is made once for every specification.
from ews_common import RESULT_COLUMNS

results = pd.DataFrame(columns=RESULT_COLUMNS)

# Run_model helper

In [ ]:
# A9: shared with 11a-d and 12a/b in ews_common.py, so a fix is made once for every specification.
from ews_common import run_model

# Training loop

In [ ]:
import optuna
import pickle
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier
from sklearn.tree import DecisionTreeClassifier
import lightgbm as lgb

# Suppress per-trial Optuna output — only warnings and errors will print
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [ ]:
# A9: shared with 11a-d and 12a/b in ews_common.py, so a fix is made once for every specification.
from ews_common import get_objective

## What's not tuned (fixed by design)


Each model has parameters that are **held fixed** — they are design choices that apply across all windows and variants, not tuning targets.

| Model | Fixed parameter | Value | Why fixed |
|-------|----------------|-------|-----------|
| All | `random_state` | `42` | Reproducibility across runs |
| LR | `solver` | `lbfgs` | Best for L2 penalty; faster and more stable than `saga` on small-medium panel data |
| LR | `penalty` | `l2` | Chosen for SHAP cross-model comparability — L1 would zero out different features per window, making SHAP values incomparable |
| LR | `max_iter` | `1000` | Sufficient for convergence; avoids `ConvergenceWarning` |
| LR, RF, DT | `class_weight` | `'balanced'` | Imbalance correction — a modelling principle, not a tuning target. Sklearn computes inverse-frequency weights automatically |
| XGBoost, LightGBM | `scale_pos_weight` | `n_neg / n_pos` | Same rationale as `class_weight`; XGB/LGBM use this scalar instead. Computed per window so it tracks the actual class ratio in each training set |
| XGBoost | `eval_metric` | `'logloss'` | Standard for binary classification with `predict_proba` |
| MLP | `activation` | `'relu'` (default) | Near-universal default for tabular data; no theoretical reason to tune it here |
| MLP | `max_iter` | `5000` | Generous budget to allow convergence on small crisis datasets without inflating wall time |
| MLP | `tol` | `1e-3` | Relaxed from sklearn default (`1e-4`) — suppresses warnings on flat loss landscapes in short inner windows; does not affect AUROC |
| MLP | `sample_weight` | `n_neg / n_pos` | MLP has no `class_weight` param — sample weights replicate the same imbalance correction used by the other models |
| DT | `criterion` | `'gini'` (default) | Gini and entropy give near-identical splits in practice; not worth a tuning dimension |
| LightGBM | `verbose` | `-1` | Suppress per-tree console output only — no effect on training |

In [ ]:
from ews_common import MODEL_NAMES, run_expanding_window

N_TRIALS = 50  # Optuna trials per model per window

# Two variants:
#   macro_standalone — full training window, no exclusions
#   rt1              — exclude crisis year (t) and post-crisis year (t+1) from training, per country.
#                      Non-crisis countries keep all their data even in crisis calendar years.
#                      Rationale: crisis/post-crisis bank profiles are "disorderly adjustments"
#                      not informative about pre-crisis vulnerability (Bussiere & Fratzscher 2006).
#                      With duration labeling, RT1 targets the ONSET year + t+1 only (not the full
#                      duration), because the sustained crisis period is itself informative.
VARIANTS = [('macro_standalone', False), ('rt1', True)]

# One feature set at country level. LR gets no country dummies here (as in the original loop),
# so it uses the same columns as the other models.
feature_sets    = {'macro': feat_cols}
feature_sets_lr = feature_sets

# A9: the loop itself (RT1 filter, inner split, Optuna tuning with the skip rule, outer refit,
# evaluation) lives in ews_common.run_expanding_window, shared by 11a-d and 12a/b, so a fix is
# made once for every specification. It returns the four objects the loop used to build here.
results, best_params_log, proba_store, fi_store = run_expanding_window(
    splits, feature_sets, feature_sets_lr, VARIANTS,
    target=target, country_col=country_col, n_trials=N_TRIALS,
    calibration=calibration,
)

for variant_name, _ in VARIANTS:
    print(f"\nMean AUROC by model and dataset ({variant_name}):")
    print(
        results[results['variant'] == variant_name]
        .groupby(['model', 'dataset'])['auroc'].mean().round(3).unstack()
    )

# LR sanity check

In [ ]:
# === LR Sanity Check ===
# Two concerns flagged during initial run:
# (1) Suspiciously high AUROC (~0.997) for LR on micro — check whether this is year-specific
#     or consistent across the evaluation window (consistent = country dummies encoding crisis membership)
# (2) L2 regularisation shrinks all coefficients toward zero — verify that financial ratios
#     carry meaningful weight and the model is not relying almost entirely on country dummies

# (1) AUROC by year for LR
lr_results = results[(results['model'] == 'logistic_regression') & (results['variant'] == 'macro_standalone')]
print("=== LR AUROC by test_year and dataset (macro_standalone) ===")
print(lr_results.pivot_table(index='test_year', columns='dataset', values='auroc').round(3).to_string())

# (2) Refit LR on the last available split and inspect coefficients
last_train, last_test = splits[-1]
cols_inspect = feat_cols

imputer_inspect        = SimpleImputer(strategy='median')
X_train_imp_inspect    = imputer_inspect.fit_transform(last_train[cols_inspect])
scaler_inspect         = StandardScaler()
X_train_scaled_inspect = scaler_inspect.fit_transform(X_train_imp_inspect)

# best_params_log key is now (variant, dataset, test_year, model)
last_year  = last_test['year'].values[0]
last_best  = best_params_log.get(('macro_standalone', 'macro', last_year, 'logistic_regression'), {})
C_inspect  = last_best.get('C', 1.0)

lr_inspect = LogisticRegression(
    solver='lbfgs', C=C_inspect,
    max_iter=1000, class_weight='balanced', random_state=7
)
lr_inspect.fit(X_train_scaled_inspect, last_train[target])

coef_df = pd.DataFrame({
    'feature': cols_inspect,
    'coef':    lr_inspect.coef_[0]
}).sort_values('coef', key=abs, ascending=False)

n_nonzero = (coef_df['coef'] != 0).sum()
print(f"\n=== LR coefficients — last window (test={last_year}), macro, C={C_inspect:.4f} ===")
print(f"Non-zero: {n_nonzero} / {len(coef_df)} features")
print(coef_df.head(20).to_string(index=False))

## Save all results

In [ ]:
import os
import pandas as pd

out_dir = "../outputs/results/12b"
os.makedirs(out_dir, exist_ok=True)

with pd.ExcelWriter(f"{out_dir}/results_12b_final.xlsx", engine="openpyxl") as writer:

    # --- Main results ---
    results.to_excel(writer, sheet_name="all_results", index=False)

    results[results["variant"] == "macro_standalone"].pivot_table(
        index="model", columns="dataset", values="auroc", aggfunc="mean"
    ).round(3).to_excel(writer, sheet_name="auroc_pivot_baseline")

    results[results["variant"] == "macro_standalone"].groupby(["model", "dataset"])[
        ["auroc", "f1", "recall", "precision", "miss_rate", "false_alarm_rate",
  "loss_08"]
    ].mean().round(3).to_excel(writer, sheet_name="mean_metrics_baseline")

    results.groupby(["variant", "model", "dataset"])["auroc"].mean().round(3)\
        .unstack("dataset").to_excel(writer, sheet_name="macro_standalone_vs_rt1")

print(f"Saved to {out_dir}/results_12b_final.xlsx")

# Save proba_store — raw (y_true, y_proba) per model/window; needed for PR/ROC curves in NB13
with open(f"{out_dir}/proba_store_final.pkl", "wb") as f:
    pickle.dump(proba_store, f)
print(f"[SAVED] {out_dir}/proba_store_final.pkl")

# Save best_params_log — hyperparameter choices per model/window; for methodology table in NB13
with open(f"{out_dir}/best_params_log_final.pkl", "wb") as f:
    pickle.dump(best_params_log, f)
print(f"[SAVED] {out_dir}/best_params_log_final.pkl")

# Convert fi_store to long-format DataFrame and save; needed for feature importance plots in NB13
fi_rows = []
for (mdl, ds, yr, var), fi in fi_store.items():
    if fi is None:
        continue
    for feat, imp in fi.items():
        fi_rows.append({"model": mdl, "dataset": ds, "test_year": yr,
                        "variant": var, "feature": feat, "importance": imp})
fi_df = pd.DataFrame(fi_rows)
fi_df.to_parquet(f"{out_dir}/fi_store_final.parquet")
print(f"[SAVED] {out_dir}/fi_store_final.parquet")

In [ ]:
import pandas as pd
df = pd.read_parquet("../data/individual_merges/parquet_saves/macro_ind_robust_me.parquet").drop(columns=['credit_gap_missing'], errors='ignore')  # final run: flag excluded, not reclassified
print(df[df["country_iso"] == "SVK"][["year", "crisis"]].sort_values("year"))